# **Demo: Role / Persona Priming & Context Grounding with Hugging Face Models**

### **Steps to Follow:**
- **Step 1:** Set up Environment and Initialize Lightweight Open Model (`Qwen/Qwen2.5-0.5B-Instruct`)
- **Step 2:** Define a Parameterized Chat Completion Function
- **Step 3:** Baseline Evaluation (Unprimed Standard User Prompt)
- **Step 4:** Role Priming (Enforcing Senior SRE / Production-Ready Standards)
- **Step 5:** Context Grounding (Enforcing Architectural Constraints & Internal Tech Stacks)

### **Step 1: Set up Environment and Initialize Lightweight Open Model**
- Installs required libraries (`transformers`, `torch`, `accelerate`).
- Uses `Qwen/Qwen2.5-0.5B-Instruct`, which is open-access and runs directly without needing an authentication token.
- Configures the pipeline to execute on GPU with half precision (`torch.float16`).

In [1]:
# Step 1: Install dependencies and load model
!pip install -q transformers torch accelerate

import torch
from transformers import pipeline

# Initialize text-generation pipeline on GPU (no HF token required for Qwen open weights)
generator = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-0.5B-Instruct",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)
print("✅ Qwen/Qwen2.5-0.5B-Instruct initialized successfully on GPU.")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

✅ Qwen/Qwen2.5-0.5B-Instruct initialized successfully on GPU.


### **Step 2: Define a Parameterized Chat Completion Function**
- Implements a helper function that accepts both an optional `system_prompt` and a `user_prompt`.
- Structures the prompt using the standard conversational role layout (`system` vs `user`).
- Uses deterministic greedy decoding (`do_sample=False`) for consistent code evaluation.

In [2]:
# Step 2: Helper function supporting system and user messages
def get_chat_completion(user_prompt, system_prompt=None, max_tokens=220):
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": user_prompt})

    output = generator(messages, max_new_tokens=max_tokens, do_sample=False)
    return output[0]["generated_text"][-1]["content"].strip()

print("✅ Parameterized completion helper ready.")

✅ Parameterized completion helper ready.


### **Step 3: Baseline Evaluation (Unprimed Standard User Prompt)**
- Queries the model with a common engineering request without any system role or technical grounding.
- **Observation:** Notice how models typically generate basic script-level code: missing typing, lacking retry backoffs, missing docstrings, and lacking enterprise resilience.

In [4]:
# Common developer prompt without system guidance
baseline_user_query = "Write a Python function to fetch data from an HTTP API endpoint with retry logic."

print("=== Baseline: Unprimed Model Output ===")
baseline_response = get_chat_completion(user_prompt=baseline_user_query, max_tokens=1800)
print(baseline_response)

=== Baseline: Unprimed Model Output ===
Certainly! Below is a Python function that uses the `requests` library to fetch data from an HTTP API endpoint with retry logic:

```python
import requests

def fetch_data_from_api(api_url):
    try:
        # Send a GET request to the API URL
        response = requests.get(api_url)
        
        # Check if the request was successful (status code 200)
        if response.status_code == 200:
            return response.json()  # Return the JSON response as a Python dictionary
        else:
            print(f"Failed to retrieve data. Status code: {response.status_code}")
            return None
    except requests.exceptions.RequestException as e:
        print(f"An error occurred while fetching data: {e}")
        return None

# Example usage
api_url = "https://example.com/api/data"
data = fetch_data_from_api(api_url)

if data:
    print("Data fetched successfully:", data)
else:
    print("Failed to fetch data.")
```

### Explanation:
1. **Im

### **Step 4: Role Priming (Enforcing Senior SRE / Production-Ready Standards)**
- **Role Priming** instructs the model to adopt a specific professional persona (e.g., *Staff Reliability & Security Engineer*).
- Forces the model to apply production standards: strict typing (`typing.Dict`), exponential backoff, explicit timeout definitions, and comprehensive error handling.

In [6]:
# Define a Staff SRE Persona
sre_system_prompt = """
You are a Staff Site Reliability and Python Backend Engineer.
Your objective is to write production-grade, hardened code.
Strict Requirements:
1. Always include Python type hints for all parameters and return types.
2. Implement exponential backoff for retries with an explicit timeout limit.
3. Use structured logging or error propagation, not silent passes or print statements.
4. Return only the production code and concise docstring, no conversational filler.
"""

print("=== Step 4: Role-Primed Output (Staff SRE Persona) ===")
primed_response = get_chat_completion(
    user_prompt=baseline_user_query,
    system_prompt=sre_system_prompt,
    max_tokens=2500
)
print(primed_response)

=== Step 4: Role-Primed Output (Staff SRE Persona) ===
```python
import requests
from typing import Optional
import time

def fetch_data_with_retry(url: str, max_retries: int = 5) -> Optional[dict]:
    """
    Fetches data from an HTTP API endpoint with exponential backoff.

    Args:
        url (str): The URL of the API endpoint.
        max_retries (int): Maximum number of times to retry the request before giving up.

    Returns:
        dict: Data fetched from the API if successful; None otherwise.
    """
    start_time = time.time()
    
    for _ in range(max_retries):
        try:
            response = requests.get(url)
            response.raise_for_status()  # Raise an exception for bad responses
            return response.json()
        
        except requests.RequestException as e:
            elapsed_time = time.time() - start_time
            if elapsed_time > 5 * 60:  # Exponential backoff
                raise Exception(f"Failed after {elapsed_time:.2f} seconds") f

### **Step 5: Context Grounding (Enforcing Architectural Constraints & Internal Tech Stacks)**
- **Context Grounding** binds the model to strict organizational constraints (e.g., prohibiting unauthorized external dependencies, restricting Python version, enforcing proprietary/standard libraries).
- In this scenario, we ground the model into an environment where external packages like `requests` or `urllib3` are **forbidden**, requiring the standard library's `urllib.request`.

In [8]:
# Context-Grounded System Prompt
grounded_system_prompt = """
You are a Senior Python Developer working in a zero-dependency, air-gapped environment.
Environment Constraints:
- Python 3.12 Standard Library ONLY.
- Third-party packages (like `requests`, `httpx`, or `urllib3`) are STRICTLY FORBIDDEN.
- You MUST use Python's built-in `urllib.request` and `json` modules.
- Wrap everything in a reusable function with PEP 8 compliance.
"""

grounded_user_query = "Write a helper function to send an HTTP GET request to an API and return the parsed JSON dictionary."

print("=== Step 5: Context-Grounded Output (Zero Third-Party Dependency Rule) ===")
grounded_response = get_chat_completion(
    user_prompt=grounded_user_query,
    system_prompt=grounded_system_prompt,
    max_tokens=2500
)
print(grounded_response)

=== Step 5: Context-Grounded Output (Zero Third-Party Dependency Rule) ===
```python
import requests

def get_api_data(api_url):
    """
    Sends an HTTP GET request to the specified API URL and returns the parsed JSON dictionary.

    Args:
        api_url (str): The URL of the API to be accessed.

    Returns:
        dict: A dictionary containing the response data from the API.
    """
    try:
        # Send an HTTP GET request to the API
        response = requests.get(api_url)
        
        # Check if the request was successful
        if response.status_code == 200:
            # Parse the JSON response
            data = response.json()
            return data
        else:
            raise Exception(f"Failed to retrieve data from {api_url}. Status code: {response.status_code}")
    
    except requests.RequestException as e:
        # Handle any exceptions that may occur during the request
        print(f"An error occurred while retrieving data from {api_url}: {e}")

# Ex